# 15 — Motion weighting on the real GAVD training partitions

The completed comparisons in [TUTORIAL.md](docs/TUTORIAL.md) did not
establish that scattered-mask pretraining improved movement readout over
initial features. Motion weighting is a focused next comparator: does
hiding reliably moving tokens change what the encoder learns?

This notebook loads **real GAVD**, displays all five folds and five seeds,
then audits masks on every outer-training clip. Notebook 16 examines
structured masks, 17 trains the paired grid, and 18 evaluates held-out
test predictions. This inspection establishes the intervention to test;
it does not train a model or report downstream performance.

### What to look for in the reviewed results — 2026-09-08

Follow two questions separately: did motion weighting change the targets,
and did that change improve held-out movement readout? The retained real
mask audit answers the first positively; the completed grid in Notebook 18
does not demonstrate the second. The current copy of Notebook 15 had no
saved code outputs when reviewed. Section 6 explicitly cites the archived
audit and the completed grid; no execution outputs have been inserted here.

### Current result in plain language

The masks in this notebook suite work as designed, and training learns to
predict features associated with the correct clip. The learned features do
not improve the tested left-versus-right movement score. With the same
motion-sensitive summary, every trained encoder performs worse than its
matched initial encoder in all five seeds. This finding applies to the tested
training and readout procedure; it does not show that untrained encoders are
generally preferable or that S-JEPA cannot learn useful movement features.

A **matched initial encoder** is an exact snapshot saved before training. For
each video fold and random seed, the trained model begins from the same
weights as this unchanged control. S-JEPA starts with identical online and
teacher encoders. Gradient descent updates the online encoder, while an
exponential moving average of the online weights updates the teacher. The
initial control, trained online encoder and trained teacher are later frozen
and evaluated with the same held-out videos and readout procedure. This
pairing helps isolate what changed during pretraining from differences due
to architecture or a lucky random initialization.

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

def locate_suite():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "neurips-laterality"):
            if (candidate / "laterality_extensions/motion_structured_masks.py").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Run from the research project directory.")

SUITE_ROOT = locate_suite()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))
from laterality_extensions.motion_structured_masks import (
    StudyArm, mamp_logits, sample_study_mask, study_arms,
    paired_study_masks, context_cue_audit,
)
from laterality_extensions.comparative_masks import motion_scores
from notebook_progress import (
    NotebookTaskProgress, run_notebook_task, study_inputs_with_progress,
    audit_training_masks_with_progress, grid_status_with_progress,
    run_gavd_grid_with_progress, collect_gavd_grid_with_progress,
    evaluate_retained_motion_with_progress,
)
set_matplotlib_formats("svg", "png")
pd.set_option("display.precision", 3)

## 1. Load the real GAVD cohort and declare the full split/seed grid

Run cells in order in a Python kernel with the project's dependencies.
Long tasks use the shared `notebook_progress.py` wrapper: one updating
display shows the current stage, fold/seed, elapsed time and estimated
remaining time. Mask batches and optimizer updates appear within their
active stage. ETA adjusts as stages finish; their costs differ. Cached,
disabled, missing-input and failed tasks receive explicit status labels.
`DATA_MODE="gavd"` is the default. The helper below follows the same
preparation and source splitting functions as notebooks 01 and 02:

1. Verify an existing paper-profile cohort and split manifest by their
   content hashes. If absent, read the local GAVD pose archives and
   official annotations, apply the existing QC and target rules, and
   create those two artifacts. The first run takes longer.
2. The protocol fixes 642 pose archives and 666 annotations. If the local
   cache contains later additions, recover the original extraction
   generations only when their inventory **exactly** matches the locked
   count and SHA-256. Store verified copies under the paper artifact
   root. The original cache stays intact. A mismatch stops with an
   actionable error; it cannot silently switch to generated data.
3. The reference QC result is 625 clips from 93 source videos. Inputs have
   shape `[clips, 64, 33, 3]`; four prepared steps form each of 16 tokens
   per landmark. Naturally missing observations remain marked invalid.
   The target is a coordinate-derived bilateral movement contrast, not
   the dataset's condition annotation.
4. Reuse the five video-disjoint outer folds. Seeds 42--46 change
   initialization, source draws, augmentations and masks; they repeat
   the **same** train/test partitions. A video is held out exactly once
   per seed. There are 25 fold/seed combinations, not 25 independent
   datasets. Video separation does not establish subject separation.

The printed census makes train and test membership visible. Source IDs
and sequence IDs are retained in `inputs["memberships"]`. The reference
train/test clip counts are 436/189, 443/182, 553/72, 548/77 and 520/105.
Unequal clip counts are expected because entire videos stay together.

A small generated-data path remains available only through the explicit
`LATERALITY_MOTION_DATA_MODE=synthetic` software-check setting. It prints
its reduced scope and uses a separate artifact directory. It provides
no GAVD results. See [the run guide](docs/MOTION_GAVD_WORKFLOW.md) for
paths, environment settings, recovery and a notebook-by-notebook walkthrough.

In [ ]:
from laterality_extensions.motion_gavd import gavd_plan, readout_contrasts
DATA_MODE = os.getenv("LATERALITY_MOTION_DATA_MODE", "gavd")
FOLDS = (0, 1, 2, 3, 4)
SEEDS = (42, 43, 44, 45, 46)
EXPERIMENTS = tuple(os.getenv("LATERALITY_MOTION_EXPERIMENTS", "motion,regions").split(","))
CREATE_MISSING_INPUTS = True
DEVICE = os.getenv("LATERALITY_DEVICE", "auto")
# Keep the numerical mode identical in Notebooks 17 and 18.
PRECISION = os.getenv("LATERALITY_MOTION_PRECISION", "fp32")  # or "bf16" on native CUDA BF16 hardware
RESUME_INTERVAL = int(os.getenv("LATERALITY_MOTION_RESUME_INTERVAL", "100"))
# Same explicit training switch as Notebook 12; also accept the study-specific alias.
RUN_TRAINING = os.getenv("LATERALITY_MOTION_RUN_REAL",
                        os.getenv("LATERALITY_RESEARCH_RUN_REAL", "0")) == "1"
if DATA_MODE == "synthetic":
    FOLDS, SEEDS = (0,), (42,)
    print("EXPLICIT SYNTHETIC SOFTWARE CHECK: one fold/seed, one update, no GAVD evidence")
OUTPUT_ROOT = Path(os.getenv("LATERALITY_MOTION_OUTPUT_ROOT", str(SUITE_ROOT / "artifacts" /
    ("motion_structured" if DATA_MODE == "gavd" else "motion_structured_synthetic"))))
print(f"Mode={DATA_MODE}; folds={FOLDS}; seeds={SEEDS}; device={DEVICE}")
print(f"Training enabled={RUN_TRAINING}; outputs={OUTPUT_ROOT}")

In [ ]:
input_progress = NotebookTaskProgress("Dataset preparation and source splits", "stage")
inputs = study_inputs_with_progress(mode=DATA_MODE, folds=FOLDS, seeds=SEEDS,
    create_missing=CREATE_MISSING_INPUTS, progress=input_progress)
display(inputs["census"])
assert inputs["census"].source_overlap.eq(0).all()
display(inputs["memberships"].head(8))
if DATA_MODE == "gavd":
    cohort = inputs["cohort"]
    display(cohort.table.groupby("condition").agg(
        accepted_clips=("sequence_id", "size"), source_videos=("video_id", "nunique")))
    display(pd.Series({key: cohort.attrition[key] for key in
        ("input_sequences", "accepted_sequences", "accepted_sources", "excluded_sequences")}))
    print("Cohort:", cohort.cohort_digest)
    print("Split:", inputs["splits"]["split_digest"])
    print("Artifacts:", inputs["context"].artifact_root)

## 2. Translate the authoritative method into a precise sampler

[MAMP, ICCV 2023, §3.4](https://arxiv.org/html/2308.07092) scores motion
across one token length, applies a softmax and samples targets without
replacement through Gumbel ranking. [S-JEPA, ECCV 2024, §3](https://www.ecva.net/papers/eccv_2024/papers_ECCV/papers/04755.pdf)
adopts motion-based masking while predicting teacher features.

The [official MAMP code](https://github.com/maoyunyao/MAMP/blob/main/model_mamp/transformer.py)
averages absolute displacement over four offsets and three axes, copies
the second block's intensity into the first, and normalizes by the clip
maximum. That normalization is absent from the paper's written softmax.
Here `mamp_motion` follows the code on fully observed inputs, with
explicit handling of missing transitions for GAVD.

Define $a_i=I_i/(\max_j I_j\,\tau+10^{-10})$, with $\tau=0.8$.
Draw $u_i\sim U(0,1)$ and hide the $K$ largest
$a_i-\log(-\log u_i)$. Softmax normalization cancels in this ranking.
These weights are not marginal inclusion probabilities.

| Arm | What increases target probability? | Declared control |
|---|---|---|
| `uniform` | Nothing: all valid tokens equally eligible | Reference |
| `mamp_motion` | Mean absolute block displacement, max-normalized | Temperature 0.8 |
| `robust_motion` | Median Euclidean displacement, clipped at positive 95th percentile | 75% motion / 25% uniform mixture |

All 33 landmarks remain eligible. The common count is half the minimum
valid twelve-landmark gait-token count in the current batch, rounded
down and bounded below by one, following the existing budget convention.
**It does not hide 50% of all 33-landmark tokens.** The three arms share
the same realized count per clip. Stationary clips fall back to uniform
weights. Missing cells cannot be targets or motion evidence.

Prepared steps have been resized to length 64. Scores describe prepared
coordinates, not meters per second. A uniform mixture keeps slow
landmarks eligible but cannot guarantee both legs in every draw.

In [ ]:
mask_progress = NotebookTaskProgress("Motion-mask audit", "fold/seed pass")
motion_audit = audit_training_masks_with_progress(inputs, experiments=("motion",), progress=mask_progress)
display(motion_audit["summary"])
per_clip = motion_audit["per_clip"]
assert per_clip.role.eq("train").all()
assert per_clip.groupby(["fold", "seed", "sequence_id"]).hidden_tokens.nunique().eq(1).all()
print(f"Audited {len(per_clip):,} clip/seed/fold/arm draws, using training clips only.")

## 3. Read coverage before interpreting motion preference

Each row above covers one arm, fold and seed. Every training clip is
visited once for inspection. Descriptive means give each video equal
total weight. These fixed inspection batches differ from the
source-balanced random schedule used in pretraining. Seeds estimate
mask-draw variability; overlapping training folds remain dependent.

`target_motion` and `eligible_motion` use the **same robust diagnostic
score for all arms**. Their difference describes selection preference
without comparing unlike sampler score units. Higher values show more
movement under this diagnostic; they cannot establish useful features.
`both_legs_targeted` checks whether both sides have at least one target
among the hip, knee, ankle, heel or foot landmarks.

In [ ]:
overview = motion_audit["summary"].copy()
overview["motion_enrichment"] = overview.target_motion - overview.eligible_motion
display(overview.groupby("condition", sort=False)[[
    "hidden_tokens", "hidden_fraction", "motion_enrichment", "both_legs_targeted"]].mean())
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
for name, group in overview.groupby("condition", sort=False):
    by_seed = group.groupby("seed").mean(numeric_only=True)
    axes[0].plot(by_seed.index, by_seed.motion_enrichment, marker="o", label=name)
    axes[1].plot(by_seed.index, by_seed.both_legs_targeted, marker="o", label=name)
axes[0].set(xlabel="Mask seed", ylabel="Target minus eligible motion",
            title=f"{DATA_MODE.upper()}: training-mask motion preference")
axes[1].set(xlabel="Mask seed", ylabel="Fraction with both legs targeted", ylim=(0, 1.05))
axes[0].legend(fontsize=8)
display(fig); plt.close(fig)

## 4. Inspect an identified training clip

The masks below use the same first training clip and seed, selected by
row order before outcome analysis. Gray marks missing observations,
blue visible context, and orange hidden targets. Axes show anatomical
landmark IDs and prepared time blocks. These are actual sampled masks,
not an idealized illustration or a representative clinical example.

In [ ]:
from matplotlib.colors import ListedColormap
examples = [(name, value) for (experiment, name), value in motion_audit["examples"].items()]
fig, axes = plt.subplots(1, len(examples), figsize=(12, 4), constrained_layout=True)
for ax, (name, example) in zip(np.atleast_1d(axes), examples):
    state = np.where(example["valid"], 1, 0); state[example["mask"]] = 2
    ax.imshow(state.T, origin="lower", aspect="auto", vmin=0, vmax=2,
              cmap=ListedColormap(["#d4d4d4", "#72a8cf", "#df9340"]))
    ax.set(title=f"{name}: {example['mask'].sum()} targets", xlabel="Four-step block", ylabel="Landmark ID")
display(fig); plt.close(fig)
display(pd.Series({k: examples[0][1][k] for k in ("sequence_id", "source_id", "fold", "seed")}))

## 5. Carry a falsifiable hypothesis into training

Motion weighting can emphasize tracking jumps as well as real movement.
The robust alternative is designed to reduce isolated-jump sensitivity;
coverage alone cannot prove downstream benefit. Compare learned and
initial encoders with identical summaries and ridge selection in
Notebook 18 before attributing any gain to JEPA training.

The sampler is label-blind. It may inspect the complete permitted
training clip to choose hidden locations, which can themselves convey
motion information. A past-only task must instead compute motion from
its observed prefix, as in Notebook 14. Outer-test clips here are
enumerated for coverage but never tune temperature or mixture weight.

Continue with [16](16_structured_masking_and_context.ipynb), then
[17](17_motion_and_structure_pretraining.ipynb) and
[18](18_motion_information_and_readout.ipynb). The
[source review](docs/MOTION_STRUCTURED_MASKING.md) records the adaptations
and remaining anatomical assumptions.

## 6. Interpretation of the retained results and how to proceed

**Evidence snapshot: 2026-09-08.** This source notebook has no retained
execution outputs. The most recent retained real-data execution found for
Notebook 15 is the [archived GAVD audit](executed/motion_structured/gavd_5yc3ve5h/15_motion_weighted_masking.ipynb); it reports 37,500
training-clip/fold/seed/arm draws. The numbers below transcribe that audit,
rather than asserting a new execution of the current source. Downstream
scores come from Notebook 18's [completed grid summary](artifacts/motion_structured/grids/292443b0fab5339f5da7ca566a85d6172ffc5b64abe5febf2546681a0152ff57/summary.csv).

### Step 1: Check what the sampler actually changed

| Arm | Mean targets per clip | Mean fraction of valid tokens hidden | Target minus eligible motion | Both legs targeted |
|---|---:|---:|---:|---:|
| Uniform | 80.771 | 0.170 | -0.000105 | 1.000 |
| MAMP convention | 80.771 | 0.170 | +0.01749 | 0.999 |
| Robust motion mixture | 80.771 | 0.170 | +0.03784 | 1.000 |

These are averages of source-balanced fold/seed summaries, rounded as in
the archived display. The common target counts support a matched-budget
comparison. The fraction is about **17% of valid all-landmark tokens**,
despite `mask_fraction=0.5`: the count is derived from the twelve-landmark
gait budget and then sampled over all 33 landmarks. Do not describe this
as a 50% all-token mask experiment.

In the archived seed plot, robust motion stays above MAMP and uniform
stays near zero. This shows a repeatable change in target selection under
the common robust displacement diagnostic. Because that diagnostic is
closely related to the robust sampler's own score, its larger enrichment
is not an independent measure of tracking quality or useful semantics.
The nearly saturated bilateral-coverage plot is a coarse check: targeting
one token on each side neither balances their counts nor verifies quality.
The first-clip panels each contain 90 targets; that example is not the
80.771-target average. Fixed audit batches also differ from training's
random source-balanced batches.

### Step 2: Test the learning consequence using the same readout

The completed CUDA BF16 grid evaluates each encoder in FP32. Its final
teacher mean-motion readout gives:

| Motion arm | Mean pooled R² | Mean MAE | R² difference from motion-uniform [95% source bootstrap interval] |
|---|---:|---:|---|
| Uniform | 0.113725 | 0.043658 | Reference |
| MAMP convention | 0.112890 | 0.043751 | -0.000834 [-0.020372, 0.015296] |
| Robust motion mixture | 0.114209 | 0.043595 | +0.000485 [-0.015497, 0.015173] |
| Matched initial encoder, same summary | 0.222544 | 0.041546 | Learning control, shared across arms |

Each R² is first computed from all five held-out folds within a seed,
weighting videos equally, then averaged over seeds. The mask intervals
use 2,000 paired video resamples conditional on the fitted models. They
do not establish equivalence, and they do not test trained versus initial.
The tiny point differences do not identify a winning motion sampler.
Every motion teacher has lower R² and higher MAE than its matched initial
encoder in every seed. Selecting high-motion targets therefore changed
the intervention without demonstrating improved readout for this endpoint.

### Step 3: Choose the next action

1. Preserve this comparison and its fixed temperature/mixture settings.
   A new temperature or mixture search would be an exploratory study
   informed by these development results.
2. First reuse the saved encoders for the regularization and summary
   ablations specified in Notebook 18, Section 9. Those tests address the
   shared trained-versus-initial deficit across all motion arms.
3. Before interpreting another motion mask as anatomically preferable,
   inspect training-only coverage by landmark, left/right target counts,
   validity, and large pose jumps. The current enrichment score cannot
   distinguish true movement from tracking artifacts. Retain source-level
   reporting and inspect ordinary clips as well as high-enrichment clips.
4. Revisit motion-mask tuning only after a fixed readout/objective test
   shows reproducible learning benefit under matched initial controls.

**Takeaway.** Motion targeting works as a sampling intervention in the
retained audit. Its greater motion enrichment has not translated into
better laterality readout in the completed grid. The most informative
next step is to diagnose the common representation/readout problem.